<a href="https://www.dna-evolutions.com/" target="_blank"><img src="https://www.dna-evolutions.com/images/dna_logo.png" width="200" title="DNA-Evolutions" alt="DNA-Evolutions"></a>

# JOpt TourOptimizer - Python REST Client Example

This notebook demonstrates how to use the JOpt TourOptimizer REST API from Python.

**Prerequisites:**
- A running TourOptimizer instance (>= 1.3.5) on `http://localhost:8081`  
  See the [Quickstart Guide](https://dna-evolutions.com/docs/getting-started/quickstart/jopt_sandboxes_quickstart) for setup instructions.
- This project installed: `pip install -e .` from the repository root.

**Documentation:** [REST Server Docs](https://dna-evolutions.com/docs/learn-and-explore/rest/rest-server-touroptimizer)

## 1. Health Check

First, verify that the TourOptimizer service is reachable.

In [10]:
from util.tour_optimizer_rest_caller import TourOptimizerRestCaller
from util.tour_optimizer_endpoints import Endpoints

# Choose the correct URL for your setup:
# - Running locally (outside Docker):  Endpoints.LOCAL_SWAGGER_TOUROPTIMIZER_URL        -> http://localhost:8081
# - Running inside the Docker sandbox:  Endpoints.LOCAL_SWAGGER_TOUROPTIMIZER_FROM_DOCKER_URL -> http://host.docker.internal:8081
TOUR_OPTIMIZER_URL = Endpoints.LOCAL_SWAGGER_TOUROPTIMIZER_FROM_DOCKER_URL

caller = TourOptimizerRestCaller(tour_optimizer_url=TOUR_OPTIMIZER_URL)

status = caller.health()
print(f"Service status: {status}")

Checking Health
'http://host.docker.internal:8081'
Service status: description=None status='UP'


## 2. Define Node and Resource Positions

Nodes are the locations to visit. Resources are the vehicles/drivers that visit them.

We use default positions in the Sydney area. You can replace these with your own coordinates.

In [11]:
from util.test_position_input import TestPositionsInput
from touroptimizer_py_client.models.position import Position

# Default sample positions (Sydney area)
node_positions = TestPositionsInput.default_sydney_node_positions()
resource_positions = TestPositionsInput.default_sydney_resource_positions()

print(f"Nodes ({len(node_positions)}):")
for pos in node_positions:
    print(f"  {pos.location_id}: ({pos.latitude}, {pos.longitude})")

print(f"\nResources ({len(resource_positions)}):")
for pos in resource_positions:
    print(f"  {pos.location_id}: ({pos.latitude}, {pos.longitude})")

Nodes (5):
  Node_0: (-34.052052, 150.668724)
  Node_1: (-34.052518, 150.709943)
  Node_2: (-34.051988, 150.71981)
  Node_3: (-34.04213, 150.729568)
  Node_4: (-34.042063, 150.739632)

Resources (4):
  Resource_0: (-34.052052, 150.668724)
  Resource_1: (-34.052518, 150.709943)
  Resource_2: (-34.051988, 150.71981)
  Resource_3: (-34.052015, 150.999808)


## 3. Build the Optimization Input

Create a `RestOptimization` object from the positions. This includes:
- Nodes with default opening hours (08:00-20:00, 5 days)
- Resources with default working hours and capacity
- A public evaluation license (max 20 elements)
- Empty connections (triggers automatic haversine distance calculation)

In [12]:
from util.test_rest_optimization_creator import TestRestOptimizationCreator

opti = TestRestOptimizationCreator.default_touroptimizer_position_test_input(
    node_positions,
    resource_positions,
    node_relations=[],
    element_connections=[],  # empty = use haversine distances
    json_license=None        # None = use public evaluation key
)

opti.ident = "JUPYTER_EXAMPLE"

print(f"Optimization input created:")
print(f"  Ident: {opti.ident}")
print(f"  Nodes: {len(opti.nodes)}")
print(f"  Resources: {len(opti.resources)}")

Optimization input created:
  Ident: JUPYTER_EXAMPLE
  Nodes: 5
  Resources: 4


## 4. Run the Optimization

Submit the optimization via `start_run` (POST /api/v1/runs), which returns a `run_id`.
The `run_id` uniquely identifies this optimization run and is essential for concurrent usage --
when multiple optimizations are running simultaneously, each run is tracked independently by its `run_id`.

The `run_id` is used to:
- Subscribe to SSE streams (progress, status, warnings, errors)
- Fetch the result via `get_run_result(run_id)`
- Check whether the optimizer has started via `get_started_signal(run_id)`

In [13]:
import touroptimizer_py_client

# Initialize the API client (caller does this lazily, but we do it explicitly here to show the run_id)
if caller.api_instance is None:
    caller._initialize_api_client()

# Step 1: Start the run -- returns a RunAcceptedResponse with the run_id
run_accepted = caller.api_instance.start_run(opti)
run_id = run_accepted.run_id

print(f"Run accepted!")
print(f"  run_id:       {run_id}")
print(f"  submitted_at: {run_accepted.submitted_at}")
print(f"  ident:        {run_accepted.ident}")
print(f"\nThe run_id uniquely identifies this run. Use it to fetch results, subscribe to streams,")
print(f"or track progress -- especially important when running multiple optimizations concurrently.")

# Step 2: Fetch the result (blocks until optimization completes)
print(f"\nWaiting for result...")
result = caller.api_instance.get_run_result(run_id)

if result:
    print("Optimization completed successfully!")
else:
    print("Optimization failed.")

'http://host.docker.internal:8081'
Run accepted!
  run_id:       d0b2ecfa-be13-4a8a-bdb6-c4b4688f6308
  submitted_at: 1775726618865
  ident:        JUPYTER_EXAMPLE

The run_id uniquely identifies this run. Use it to fetch results, subscribe to streams,
or track progress -- especially important when running multiple optimizations concurrently.

Waiting for result...
Optimization completed successfully!


## 5. Inspect the Result

The result contains the optimized routes as a text solution and detailed route data.

In [14]:
if result and result.extension:
    # Print the human-readable text solution
    text_solution = result.extension.text_solution
    if text_solution:
        print("=== Text Solution ===")
        print(text_solution.text_solution)
else:
    print("No result available.")

=== Text Solution ===

-----------------------------------------------------------
---------------------- RUN --------------------------------
-----------------------------------------------------------
 Optimization-IDENT      : JUPYTER_EXAMPLE

-----------------------------------------------------------
--------------------- RESULTS -----------------------------
-----------------------------------------------------------
 Number of Route         : 20
 Number of Route (sched.): 3
 Total Route Elements    : 9
 Cost                    : 17.258318335477234
-----------------------------------------------------------
 Total time        [h]   : 2
 Total idle time   [h]   : 0
 Total prod. time  [h]   : 2
 Total tran. time  [h]   : 0
 Total utilization [%]   : 97
 Total distance    [km]  : 4
 Termi. time       [h]   : 0
 Termi. distance   [km]  : 1

-----------------------------------------------------------
--------------------- ROUTES ------------------------------
-------------------------

## 6. Inspect Routes and Assignments

Examine which nodes were assigned to which resources, and the schedule for each route.

In [15]:
if result and result.solution and result.solution.routes:
    for route in result.solution.routes:
        print(f"\n--- Route for resource: {route.resource_id} ---")
        print(f"    Start: {route.start_time}  |  Elements: {len(route.element_details)}")
        for detail in route.element_details:
            print(f"  -> {detail.element_id}  (arrive: {detail.arrival_time}, depart: {detail.departure_time})")
else:
    print("No routes in result.")


--- Route for resource: Resource_1 ---
    Start: 2030-01-01 08:00:00+00:00  |  Elements: 1
  -> Node_1  (arrive: 2030-01-01 08:00:00+00:00, depart: 2030-01-01 08:30:00+00:00)

--- Route for resource: Resource_1 ---
    Start: 2030-01-02 08:00:00+00:00  |  Elements: 0

--- Route for resource: Resource_1 ---
    Start: 2030-01-03 08:00:00+00:00  |  Elements: 0

--- Route for resource: Resource_1 ---
    Start: 2030-01-04 08:00:00+00:00  |  Elements: 0

--- Route for resource: Resource_1 ---
    Start: 2030-01-05 08:00:00+00:00  |  Elements: 0

--- Route for resource: Resource_2 ---
    Start: 2030-01-01 08:00:00+00:00  |  Elements: 3
  -> Node_2  (arrive: 2030-01-01 08:00:00+00:00, depart: 2030-01-01 08:30:00+00:00)
  -> Node_4  (arrive: 2030-01-01 08:31:36.922000+00:00, depart: 2030-01-01 09:01:36.922000+00:00)
  -> Node_3  (arrive: 2030-01-01 09:02:19.044000+00:00, depart: 2030-01-01 09:32:19.044000+00:00)

--- Route for resource: Resource_2 ---
    Start: 2030-01-02 08:00:00+00:00  

## 7. Export to JSON

Save the optimization input or result to a JSON file for later use or inspection.

In [16]:
import json

# Export the input
input_json = TourOptimizerRestCaller.to_json(opti)
print(f"Input JSON (first 500 chars):\n{input_json[:500]}...")

# Save to file
# TourOptimizerRestCaller.save_to_json_file(opti, "optimization_input.json")
# print("Saved to optimization_input.json")

Input JSON (first 500 chars):
{
    "ident": "JUPYTER_EXAMPLE",
    "nodes": [
        {
            "id": "Node_0",
            "type": {
                "typeName": "Geo",
                "position": {
                    "latitude": -34.052052,
                    "longitude": 150.668724,
                    "locationId": "Node_0"
                }
            },
            "openingHours": [
                {
                    "begin": "2030-01-01T08:00:00",
                    "end": "2030-01-01T20:00:00",
           ...


## 8. Custom Positions (Optional)

Replace the default positions with your own coordinates.

In [17]:
# Example: Custom positions in Berlin
custom_nodes = [
    Position(latitude=52.5200, longitude=13.4050, locationId="Berlin_Center"),
    Position(latitude=52.5075, longitude=13.3903, locationId="Brandenburg_Gate"),
    Position(latitude=52.5163, longitude=13.3777, locationId="Tiergarten"),
    Position(latitude=52.4934, longitude=13.4473, locationId="Kreuzberg"),
]

custom_resources = [
    Position(latitude=52.5200, longitude=13.4050, locationId="Driver_1"),
    Position(latitude=52.4934, longitude=13.4473, locationId="Driver_2"),
]

custom_opti = TestRestOptimizationCreator.default_touroptimizer_position_test_input(
    custom_nodes, custom_resources, [], [], None
)
custom_opti.ident = "JUPYTER_BERLIN_EXAMPLE"

print(f"Custom optimization: {len(custom_nodes)} nodes, {len(custom_resources)} resources")

# Uncomment to run:
# custom_result = caller.optimize(custom_opti)
# if custom_result and custom_result.extension and custom_result.extension.text_solution:
#     print(custom_result.extension.text_solution.text_solution)

Custom optimization: 4 nodes, 2 resources
